[Reference](v)

# 1. Columnar Storage Over Row-Based


In [1]:
import pandas as pd
import time
import os

# Generate sample data
df = pd.DataFrame({
    'user_id': range(1000000),
    'event': ['click'] * 1000000,
    'metadata': ['json_payload'] * 1000000
})
# Row-based: CSV reads entire file even for 2 columns
df.to_csv('logs.csv', index=False)
start = time.time()
df_csv = pd.read_csv('logs.csv', usecols=['user_id', 'event'])
csv_time = time.time() - start
# Columnar: Parquet reads only required column chunks
df.to_parquet('logs.parquet', engine='pyarrow', compression='snappy')
start = time.time()
df_parquet = pd.read_parquet('logs.parquet', columns=['user_id', 'event'])
parquet_time = time.time() - start
print(f"Speedup: {csv_time/parquet_time:.1f}x")
print(f"Size reduction: {os.path.getsize('logs.csv')/os.path.getsize('logs.parquet'):.1f}x")

Speedup: 3.8x
Size reduction: 6.0x


# 2. Predicate Pushdowns


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .config("spark.sql.parquet.filterPushdown", "true") \
    .getOrCreate()
# Without pushdown: Reads all data then filters (slow)
df_slow = spark.read.parquet("s3://bucket/logs/") \
    .filter(col("date") == "2024-01-01")
# With pushdown: Filter evaluated at storage layer (fast)
# Parquet footer statistics skip row groups where max(date) < 2024-01-01
df_fast = spark.read.parquet("s3://bucket/logs/") \
    .filter(col("date") == "2024-01-01")
# Verify pushdown in physical plan
df_fast.explain()
# Output shows: FileScan parquet [date=2024-01-01] (PushedFilters: [IsNotNull(date), EqualTo(date,2024-01-01)])

# 3. Partition Pruning


In [6]:
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.getOrCreate()
# # Create Hive-style partitioned dataset
# df = spark.range(1000000).withColumn("date", (col("id") % 30).cast("string"))
# df.write.partitionBy("date").parquet("s3://bucket/events/")
# # Query triggers pruning: only scans date=1 and date=2 directories
# result = spark.read.parquet("s3://bucket/events/") \
#     .filter(col("date").isin(["1", "2"])) \
#     .select("id")
# # Monitor: spark.sparkContext.uiWebUrl shows "number of files read: 2" vs 30

# 4. Dictionary Encoding


In [5]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

# Data with low cardinality (repeating strings)
data = {
    'status': ['active'] * 500000 + ['inactive'] * 500000,
    'user_id': range(1000000)
}
table = pa.Table.from_pydict(data)
# Write with dictionary encoding (default for strings)
pq.write_table(
    table,
    'encoded.parquet',
    use_dictionary=True,
    dictionary_pagesize_limit=1048576
)
# Write without dictionary
pq.write_table(table, 'plain.parquet', use_dictionary=False)
# Verify compression
import os
enc_size = os.path.getsize('encoded.parquet')
plain_size = os.path.getsize('plain.parquet')
print(f"Compression ratio: {plain_size/enc_size:.1f}x")  # Output: 2.5x

Compression ratio: 1.1x


# 5. Run-Length Encoding (RLE)


In [7]:
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

# Create run-length friendly data (sorted timestamps with runs)
timestamps = np.repeat(
    pd.date_range('2024-01-01', periods=1000, freq='min'),
    1000  # 1000 identical timestamps per minute
)
df = pa.table({
    'timestamp': timestamps,
    'event_id': range(1000000)
})
# Sort to maximize RLE effectiveness
df = df.sort_by('timestamp')
# Parquet automatically applies RLE to timestamp column
pq.write_table(df, 'rle_compressed.parquet', use_dictionary=False)
# Check file size
import os
size_mb = os.path.getsize('rle_compressed.parquet') / (1024 * 1024)
print(f"RLE compressed size: {size_mb:.2f} MB")  # ~2MB vs 20MB uncompressed

RLE compressed size: 4.19 MB


# 6. Delta Lake for ACID


In [9]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

builder = SparkSession.builder \
    .appName("DeltaDemo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
spark = configure_spark_with_delta_pip(builder).getOrCreate()
# Create initial Delta table
df = spark.range(0, 1000).withColumn("value", col("id") * 2)
df.write.format("delta").save("/tmp/delta-table")
# ACID merge (upsert) without rewriting entire table
delta_table = DeltaTable.forPath(spark, "/tmp/delta-table")
updates = spark.range(500, 1500).withColumn("value", col("id") * 3)
delta_table.alias("target").merge(
    updates.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
# Query previous version (time travel)
previous_version = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/tmp/delta-table")

#

# 7. Z-Ordering for Locality


In [10]:
from delta.tables import DeltaTable

# Create sample Delta table with scattered data
df = spark.range(0, 1000000).select(
    (col("id") % 10000).alias("user_id"),
    (col("id") % 100).alias("event_type"),
    col("id")
)
df.write.format("delta").mode("overwrite").save("/tmp/events")
# Apply Z-Ordering on frequently filtered columns
spark.sql("""
    OPTIMIZE delta.`/tmp/events`
    ZORDER BY (user_id, event_type)
""")
# Query benefits from locality: touches 5 files instead of 50
result = spark.read.format("delta").load("/tmp/events") \
    .filter((col("user_id") == 500) & (col("event_type") == 25))
result.explain()  # Shows file pruning statistics

# 8. Materialized Views


In [12]:
import pandas as pd
from datetime import datetime

# Base table: 100M rows (expensive to scan)
base_df = pd.read_parquet('events.parquet')
# Materialize expensive aggregation (run nightly via Airflow)
materialized = base_df.groupby(['date', 'category']).agg({
    'revenue': 'sum',
    'unique_users': pd.Series.nunique,
    'avg_session_duration': 'mean'
}).reset_index()
# Add refresh timestamp for monitoring
materialized['last_refresh'] = datetime.now()
# Write to fast storage for BI tools
materialized.to_parquet(
    'mv_daily_metrics.parquet',
    partition_cols=['date'],
    engine='pyarrow'
)
# Dashboard queries read 10K rows instead of 100M
dashboard = pd.read_parquet(
    'mv_daily_metrics.parquet',
    columns=['date', 'revenue', 'category']
)

# 9. Approximate Queries


In [13]:
from datasketch import HyperLogLog
import pandas as pd
import time

# Generate large dataset
df = pd.DataFrame({'user_id': range(10000000)})
# Exact count (slow, full scan)
start = time.time()
exact_count = df['user_id'].nunique()
exact_time = time.time() - start
# Approximate count using HyperLogLog (fast)
start = time.time()
hll = HyperLogLog(p=8)  # 1.04% standard error, 2.56KB memory
for user_id in df['user_id']:
    hll.update(str(user_id).encode('utf8'))
approx_count = len(hll)  # O(1) operation
approx_time = time.time() - start
error_rate = abs(exact_count - approx_count) / exact_count
print(f"Exact: {exact_count} ({exact_time:.2f}s)")
print(f"Approx: {approx_count} ({approx_time:.2f}s)")
print(f"Speedup: {exact_time/approx_time:.0f}x, Error: {error_rate:.2%}")

# 10. Vectorized Execution


In [15]:
import pyarrow as pa
import pyarrow.compute as pc
import time
import pandas as pd
import numpy as np

# Create large dataset
size = 10_000_000
df_pandas = pd.DataFrame({
    'price': np.random.randn(size),
    'tax_rate': np.full(size, 0.08)
})
# Slow: Python row-by-row iteration
start = time.time()
result_slow = [row['price'] * row['tax_rate'] for _, row in df_pandas.iterrows()]
slow_time = time.time() - start
# Fast: Arrow vectorized execution using SIMD
table = pa.Table.from_pandas(df_pandas)
start = time.time()
result_fast = pc.multiply(table['price'], table['tax_rate'])
fast_time = time.time() - start
print(f"Vectorized speedup: {slow_time/fast_time:.0f}x")
# Zero-copy conversion to Pandas (no serialization overhead)
df_result = result_fast.to_pandas()  # Shared memory buffer, instant